# 03 — LLM Earnings Surprise: Prompting, Scoring & Comparisons

**NTAM research pipeline | Input: Notebook 02 WRDS Q4 2025 event exports**

This notebook performs the *separate* LLM inference step: loads `earnings_event_transcripts.jsonl` and `earnings_events.csv`, scores transcripts with two fixed prompts, records structured scores, and exports event-level results. It **does not** estimate R² or train a return predictor; those belong to later notebooks after the SUE/returns merge.

**Research cautions:** The prompts below are **NTAM experimental baselines, not verified verbatim prompts from Koijen–Levy**. A transcript may have been created after the initial earnings announcement/price reaction. Scores are suitable for exploratory text analysis; use them in return regressions or backtests only after validating timestamps, security links and information availability. Do not optimize prompts using the held-out evaluation period's returns.

**Default behavior:** `DRY_RUN=True` (no API calls), `PILOT_EVENTS=5`, two prompts, one model. Review the preview, set an API key locally, then switch `DRY_RUN=False` to make calls. Inference may incur API charges. Never commit secrets or licensed transcript text to Git.

## 0. Dependencies and paths

In [1]:
# Run once if necessary, then restart the kernel:
# %pip install -U openai pandas
from pathlib import Path
from datetime import datetime, timezone
import os, json, time, hashlib, re
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'processed' / 'earnings_events_q4_2025'
OUT_DIR = ROOT / 'data' / 'processed' / 'llm_earnings_scores_q4_2025'
OUT_DIR.mkdir(parents=True, exist_ok=True)
EVENTS_CSV = DATA_DIR / 'earnings_events.csv'
TRANSCRIPTS_JSONL = DATA_DIR / 'earnings_event_transcripts.jsonl'
assert EVENTS_CSV.is_file(), f'Missing {EVENTS_CSV}. Run Notebook 02.'
assert TRANSCRIPTS_JSONL.is_file(), f'Missing {TRANSCRIPTS_JSONL}. Run Notebook 02.'
print('Input:', DATA_DIR.resolve())
print('Output:', OUT_DIR.resolve())

Input: C:\Users\vidhi\OneDrive\Desktop\Project lab\Autumn 26\LLM-based-earnings-surprise_NTAM\data\processed\earnings_events_q4_2025
Output: C:\Users\vidhi\OneDrive\Desktop\Project lab\Autumn 26\LLM-based-earnings-surprise_NTAM\data\processed\llm_earnings_scores_q4_2025


## 1. Experiment configuration

Change `PILOT_EVENTS=None` only after inspecting pilot outputs. Model IDs depend on your API access. The first run should remain dry.

In [14]:
DRY_RUN = False             # Set False only when ready to use the paid API
PILOT_EVENTS = 5           # None = all eligible events
MODELS = ['gpt-4.1-mini']  # Add a second model only after the pilot succeeds
PROMPTS = ['simple_v1', 'structured_v1']
TRANSCRIPT_VERSION = 'first' # 'first' preferred for earlier availability; 'final' exploratory only
MIN_WORDS = 1000
MAX_INPUT_CHARS = 90000   # Long transcripts truncated; logged explicitly
MAX_API_CALLS = 20        # Hard per-run safety cap; raise deliberately for full universe
MAX_RETRIES = 3
SLEEP_SECONDS = 0.3
TEMPERATURE = 0
assert TRANSCRIPT_VERSION in ('first','final')
print('Planned models:', MODELS, '| prompts:', PROMPTS, '| dry run:', DRY_RUN)

Planned models: ['gpt-4.1-mini'] | prompts: ['simple_v1', 'structured_v1'] | dry run: False


## 2. Read Notebook 02 exports and check event coverage

In [15]:
metadata = pd.read_csv(EVENTS_CSV, dtype={'keydevid':'Int64'})
assert metadata['keydevid'].notna().all() and metadata['keydevid'].is_unique, 'Event IDs must be unique'
print('Events in Notebook 02 export:', len(metadata))
print('Unique companies:', metadata['companyid'].nunique())
print('Call-date range:', metadata['call_date'].min(), 'to', metadata['call_date'].max())
if len(metadata) <= 50:
    print('NOTE: This looks like a pilot-sized export. To expand the universe, rerun Notebook 02 with PILOT_ONLY=False, MAX_EVENTS=None.')
display(metadata[['keydevid','companyname','call_date','first_word_count','needs_review']].head(10))

Events in Notebook 02 export: 5
Unique companies: 5
Call-date range: 2025-10-14 to 2025-12-10
NOTE: This looks like a pilot-sized export. To expand the universe, rerun Notebook 02 with PILOT_ONLY=False, MAX_EVENTS=None.


,keydevid,companyname,call_date,first_word_count,needs_review
0,1880672021,JPMorgan Chase & Co.,2025-10-14,9007,False
1,1962423929,The Coca-Cola Company,2025-10-21,9347,False
2,1958574325,Caterpillar Inc.,2025-10-29,9171,False
3,1964399669,Microsoft Corporation,2025-10-29,8698,False
4,1972492780,Oracle Corporation,2025-12-10,6867,False


## 3. Load and validate transcript texts

Read JSONL one event at a time and retain only events in the selected sample. Do not put full licensed transcripts in the scored CSV.

In [16]:
eligible = metadata.copy()
word_col = f'{TRANSCRIPT_VERSION}_word_count'
text_col = f'{TRANSCRIPT_VERSION}_text'
created_col = f'{TRANSCRIPT_VERSION}_created_at_utc'
eligible[word_col] = pd.to_numeric(eligible[word_col], errors='coerce')
eligible = eligible[eligible[word_col].ge(MIN_WORDS)].sort_values(['call_date','keydevid'])
if PILOT_EVENTS is not None:
    eligible = eligible.head(PILOT_EVENTS)
selected_ids = set(eligible['keydevid'].astype(int))
transcripts = {}
with TRANSCRIPTS_JSONL.open('r', encoding='utf-8') as fh:
    for line_number, line in enumerate(fh, 1):
        if not line.strip(): continue
        obj = json.loads(line)
        event_id = int(obj['keydevid'])
        if event_id in selected_ids:
            if event_id in transcripts: raise ValueError(f'Duplicate transcript event ID {event_id}')
            transcripts[event_id] = obj
missing = selected_ids - transcripts.keys()
if missing: raise ValueError(f'Missing JSONL transcript records for {sorted(missing)[:10]}')
eligible = eligible[eligible['keydevid'].astype(int).isin(transcripts)]
print('Eligible selected events:', len(eligible))
print('Potential API calls:', len(eligible)*len(MODELS)*len(PROMPTS))
if len(eligible)==0: raise ValueError('No eligible transcripts; inspect Notebook 02 outputs and MIN_WORDS.')
display(eligible[['keydevid','companyname','call_date',word_col,created_col]].head(10))

Eligible selected events: 5
Potential API calls: 10


,keydevid,companyname,call_date,first_word_count,first_created_at_utc
0,1880672021,JPMorgan Chase & Co.,2025-10-14,9007,2025-10-14 16:17:46+00:00
1,1962423929,The Coca-Cola Company,2025-10-21,9347,2025-10-21 13:53:59+00:00
2,1958574325,Caterpillar Inc.,2025-10-29,9171,2025-10-29 13:44:54+00:00
3,1964399669,Microsoft Corporation,2025-10-29,8698,2025-10-29 22:46:03+00:00
4,1972492780,Oracle Corporation,2025-12-10,6867,2025-12-10 23:00:12+00:00


## 4. Fixed prompts and machine-readable output

**Do not ask the LLM to predict the observed stock return, and do not supply future prices or SUE labels.** These prompts measure text-implied business developments, *not true consensus-adjusted earnings surprises* without an external expectations benchmark. The `overall_signal` is a standardized judgment on [-1, 1], not an observed earnings surprise.

In [17]:
PROMPT_TEXT = {
'simple_v1': """Read the earnings-call transcript as a contemporaneous analyst. Assess the overall business outlook and management commentary conveyed in this transcript. Use only information in the supplied text. Do NOT assume knowledge of market reaction, later outcomes, consensus EPS, or external information. If expectations are not explicitly stated, do not claim a measured beat/miss. Assign an overall_signal from -1 (strongly negative outlook) through 0 (mixed/neutral) to +1 (strongly positive outlook). Provide confidence 0 to 1, a concise rationale, and up to 3 brief direct evidence excerpts. Return JSON only with keys overall_signal, confidence, rationale, evidence. Do not include any other keys.""",
'structured_v1': """You are analyzing an earnings-call transcript using ONLY its contents. Separate management commentary on revenue/demand, margins/costs, forward guidance, and risks. Scores represent positive/negative *information in the call*, not verified consensus beats or realized returns. Do not use knowledge of subsequent stock prices, news, or unstated analyst expectations. Assign each dimension a score from -1 to +1 or null if insufficient evidence. Assign overall_signal from -1 to +1, confidence from 0 to 1, concise rationale, and up to 3 short direct evidence excerpts. Return JSON only with keys overall_signal, confidence, revenue_demand, margins_costs, guidance_outlook, risks, rationale, evidence. A positive risks score means risk outlook is improving. Do not include any other keys."""
}
SYSTEM_PROMPT = 'You are a careful financial-text research assistant. Only use supplied source text. Return valid JSON and never fabricate evidence or consensus expectations.'

def make_user_prompt(row, transcript, prompt_id):
    return (f'Company: {row.companyname}\nCall date (not necessarily announcement date): {row.call_date}\n'
            f'Transcript version: {TRANSCRIPT_VERSION}\n\nINSTRUCTIONS:\n{PROMPT_TEXT[prompt_id]}\n\n'
            f'TRANSCRIPT:\n{transcript}')

for pid in PROMPTS: assert pid in PROMPT_TEXT
print('Prompt IDs:', list(PROMPT_TEXT))

Prompt IDs: ['simple_v1', 'structured_v1']


## 5. Quality/timing flags and input preparation

The source timestamp is a **creation timestamp**, not independently verified publication time. The notebook preserves that distinction. Input truncation is explicit, never silent.

In [18]:
def prepare_input(row):
    obj = transcripts[int(row.keydevid)]
    full = obj.get(text_col) or ''
    if not full.strip(): raise ValueError('Empty transcript')
    truncated = len(full) > MAX_INPUT_CHARS
    text = full[:MAX_INPUT_CHARS] if truncated else full
    created = pd.to_datetime(getattr(row, created_col), utc=True, errors='coerce')
    call_date = pd.to_datetime(row.call_date, errors='coerce')
    # Day-level flag only: we do NOT know the precise release/publication time.
    created_after_call_day = bool(pd.notna(created) and pd.notna(call_date) and created.date() > call_date.date())
    return text, {
        'transcript_char_count':len(full), 'sent_char_count':len(text),
        'truncated':truncated, 'transcript_created_at_utc': str(getattr(row, created_col)),
        'created_after_call_day':created_after_call_day,
        'availability_verified':False,
        'transcript_sha256':hashlib.sha256(full.encode('utf-8')).hexdigest(),
    }

preview = []
for row in eligible.itertuples(index=False):
    t, flags = prepare_input(row)
    preview.append({'keydevid':int(row.keydevid),'companyname':row.companyname,**flags})
preview_df = pd.DataFrame(preview)
display(preview_df)
print('Truncated transcripts:', int(preview_df.truncated.sum()))
print('Transcript publication/availability independently verified: NO')

,keydevid,companyname,transcript_char_count,sent_char_count,truncated,transcript_created_at_utc,created_after_call_day,availability_verified,transcript_sha256
0,1880672021,JPMorgan Chase & Co.,53267,53267,False,2025-10-14 16:17:46+00:00,False,False,e7525b780e5c94cfd38da0e52165a0920c9e082fd11a90...
1,1962423929,The Coca-Cola Company,56793,56793,False,2025-10-21 13:53:59+00:00,False,False,95293dea0bb70d5738ab5591edf316e882798101763e88...
2,1958574325,Caterpillar Inc.,56202,56202,False,2025-10-29 13:44:54+00:00,False,False,4505ab7a23d7b79cb97a96a9302792b841db9ab282a9bf...
3,1964399669,Microsoft Corporation,53476,53476,False,2025-10-29 22:46:03+00:00,False,False,c3aa8147f26f19fa916c358fa36abead29b4d35c1348d8...
4,1972492780,Oracle Corporation,41637,41637,False,2025-12-10 23:00:12+00:00,False,False,596bd9ac8701e1d3499b68435ea51cc307f77de6910e57...


Truncated transcripts: 0
Transcript publication/availability independently verified: NO


## 6. Secure API setup and JSON validation

Set `OPENAI_API_KEY` in your local environment, **not** in the notebook. You may enter it with `getpass` for the current kernel. The API is called only when `DRY_RUN=False`. API model IDs may change. No API usage occurs in dry-run mode.

In [29]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [30]:
from getpass import getpass
if not DRY_RUN:
    if not os.environ.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key (hidden; current kernel only): ')
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], timeout=120)
else:
    client = None
    print('DRY RUN: no API credentials required, no paid requests.')

DIMENSIONS = ['revenue_demand','margins_costs','guidance_outlook','risks']
def validate_response(raw, prompt_id):
    if not isinstance(raw, dict): raise ValueError('Model response is not a JSON object')
    required = ['overall_signal','confidence','rationale','evidence']
    if prompt_id == 'structured_v1': required += DIMENSIONS
    missing = [k for k in required if k not in raw]
    if missing: raise ValueError(f'Missing keys: {missing}')
    def number(key, low, high, allow_null=False):
        val=raw.get(key)
        if val is None and allow_null: return None
        if isinstance(val,bool) or not isinstance(val,(int,float)) or not low <= val <= high:
            raise ValueError(f'{key} must be numeric in [{low},{high}]' + (' or null' if allow_null else ''))
        return float(val)
    clean={'overall_signal':number('overall_signal',-1,1),
           'confidence':number('confidence',0,1)}
    for dim in DIMENSIONS:
        clean[dim] = number(dim,-1,1,allow_null=True) if prompt_id=='structured_v1' else None
    if not isinstance(raw['rationale'],str): raise ValueError('rationale must be string')
    if not isinstance(raw['evidence'],list) or not all(isinstance(x,str) for x in raw['evidence']):
        raise ValueError('evidence must be list of strings')
    clean['rationale']=raw['rationale'][:2500]
    clean['evidence']=raw['evidence'][:3]
    return clean

# Local validator smoke test; not an LLM result
assert validate_response({'overall_signal':0.2,'confidence':0.6,'rationale':'test','evidence':[]},'simple_v1')['overall_signal']==0.2
print('Response validation ready.')

Response validation ready.


## 7. Run inference with retries, checkpoints and call limits

Successful results and errors are appended to separate JSONL files. Re-running skips already successful `(event, model, prompt, transcript hash)` combinations. The model is not allowed to see future returns. If you change the prompt text, increment its version (`simple_v2`, etc.) before rerunning. **Never commit these outputs without checking licensing.**

In [31]:
RESULTS_FILE = OUT_DIR / 'llm_scores_raw.jsonl'
ERRORS_FILE = OUT_DIR / 'llm_scoring_errors.jsonl'

def load_success_keys(path):
    keys=set()
    if path.exists():
        with path.open('r',encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    d=json.loads(line)
                    if d.get('status')=='ok':
                        keys.add((int(d['keydevid']),d['model'],d['prompt_id'],d['transcript_version'],d['transcript_sha256']))
                except (ValueError,KeyError,TypeError): pass
    return keys

def append_jsonl(path, obj):
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(obj,ensure_ascii=False,allow_nan=False,default=str)+'\n')

def call_llm(model, prompt_id, user_prompt):
    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        response_format={'type':'json_object'},
        messages=[{'role':'system','content':SYSTEM_PROMPT},
                  {'role':'user','content':user_prompt}],
    )
    content=response.choices[0].message.content
    return json.loads(content), {'input_tokens':getattr(response.usage,'prompt_tokens',None),
                                 'output_tokens':getattr(response.usage,'completion_tokens',None)}

existing = load_success_keys(RESULTS_FILE)
planned = []
for row in eligible.itertuples(index=False):
    transcript, flags=prepare_input(row)
    for model in MODELS:
        for prompt_id in PROMPTS:
            key=(int(row.keydevid),model,prompt_id,TRANSCRIPT_VERSION,flags['transcript_sha256'])
            if key not in existing:
                planned.append((row,transcript,flags,model,prompt_id))
print('Successful existing combinations:',len(existing))
print('New API calls needed:',len(planned))
if DRY_RUN:
    print('DRY RUN: preview only. Set DRY_RUN=False and rerun cells 6–8 to score.')
    display(pd.DataFrame([{'company':r.companyname,'keydevid':int(r.keydevid),'model':m,'prompt':p,'chars':len(t)}
                          for r,t,f,m,p in planned]).head(20))
else:
    if len(planned)>MAX_API_CALLS:
        raise RuntimeError(f'{len(planned)} calls exceed MAX_API_CALLS={MAX_API_CALLS}. Increase limit deliberately or reduce pilot size.')
    success=0; failed=0
    for i,(row,transcript,flags,model,prompt_id) in enumerate(planned,1):
        base={'keydevid':int(row.keydevid),'companyid':int(row.companyid),
              'companyname':str(row.companyname),'call_date':str(row.call_date),
              'model':model,'prompt_id':prompt_id,'transcript_version':TRANSCRIPT_VERSION,
              **flags,'scored_at_utc':datetime.now(timezone.utc).isoformat()}
        prompt=make_user_prompt(row,transcript,prompt_id)
        last_error=None
        for attempt in range(1,MAX_RETRIES+1):
            try:
                raw,usage=call_llm(model,prompt_id,prompt)
                clean=validate_response(raw,prompt_id)
                append_jsonl(RESULTS_FILE,{**base,**clean,**usage,'status':'ok'})
                success+=1;last_error=None
                break
            except Exception as exc:
                last_error=f'{type(exc).__name__}: {str(exc)[:400]}'
                if attempt<MAX_RETRIES: time.sleep(min(2**attempt,10))
        if last_error is not None:
            append_jsonl(ERRORS_FILE,{**base,'status':'error','error':last_error})
            failed+=1
        print(f'{i}/{len(planned)} | {row.companyname} | {model} | {prompt_id} | {"OK" if last_error is None else "ERROR"}')
        time.sleep(SLEEP_SECONDS)
    print('New successes:',success,'| new failures:',failed)

Successful existing combinations: 0
New API calls needed: 10
1/10 | JPMorgan Chase & Co. | gpt-4.1-mini | simple_v1 | ERROR


KeyboardInterrupt: 

## 8. Export a tidy scoring table and compare prompt coverage

No fabricated scores: in dry-run mode, this step reports that scoring has not occurred. Each event × model × prompt has its own row. Evidence/rationale are kept as research audit fields, but the raw transcript text is not copied into this CSV.

In [ ]:
if RESULTS_FILE.exists():
    with RESULTS_FILE.open('r',encoding='utf-8') as f:
        scored=[json.loads(line) for line in f if line.strip()]
    scored_df=pd.DataFrame(scored)
    scored_df=scored_df[scored_df['status'].eq('ok')].copy()
    scored_df=scored_df.sort_values('scored_at_utc').drop_duplicates(
        ['keydevid','model','prompt_id','transcript_version','transcript_sha256'],keep='last')
    scored_df['evidence_json']=scored_df['evidence'].apply(lambda x:json.dumps(x,ensure_ascii=False))
    scored_df=scored_df.drop(columns=['evidence'])
    score_csv=OUT_DIR/'llm_earnings_scores.csv'
    scored_df.to_csv(score_csv,index=False)
    print('Saved:',score_csv,'| rows:',len(scored_df))
    display(scored_df[['companyname','model','prompt_id','overall_signal','confidence','truncated','created_after_call_day']].head(20))
    print('Counts by model/prompt:')
    display(scored_df.groupby(['model','prompt_id']).size().rename('n').reset_index())
else:
    print('No scores yet. This is expected while DRY_RUN=True. No fake values were generated.')

## 9. Handoff to Notebook 04 and research checklist

**Main export:** `data/processed/llm_earnings_scores_q4_2025/llm_earnings_scores.csv`.

**Join keys:** `keydevid` (CIQ event ID), `companyid` (CIQ company ID), `call_date`; verify a valid GVKEY/security mapping before joining Compustat, I/B/E/S or stock returns. Keep `model`, `prompt_id`, and `transcript_version` separate when comparing experiments.

**Before claiming an R² result or tradable alpha:**

1. Validate that the **earnings announcement timestamp**, **call timestamp**, **transcript publication timestamp**, and **return window** are aligned. Transcript creation alone is insufficient proof of availability.
2. Keep SUE as a separate measured numerical variable; an LLM's qualitative signal is **not** an EPS consensus surprise.
3. Decide the benchmark return window and control for market movement; run SUE-only and SUE+LLM regressions in Notebook 04/05.
4. Freeze prompt designs before out-of-sample evaluation. Avoid optimizing prompts on Q4 2025 outcomes if that quarter is the test set.
5. Evaluate sample selection, missing transcripts, truncation, and repeatability. Multiple-model comparison is optional and should hold prompt definitions fixed.
6. Check WRDS/CIQ licensing and approved API handling of transcript text before uploading to any third-party model provider.

**Scaling:** Notebook 03 can only score what Notebook 02 exported. To get the full Q4 2025 universe, first rerun Notebook 02 with `PILOT_ONLY=False`, `MAX_EVENTS=None`, then set `PILOT_EVENTS=None` here and deliberately raise `MAX_API_CALLS` after estimating cost. Do not run a full-universe paid job without a budget review.